# Deliverable 2
2026-06-21

Begin test assessments and complete Group 1 "required" tests.

| Stage | Stage specifications | Time required | Start-End |
| ----- | -------------------- | ------------- | --------- |
| 1 | Learn about the community and note shortcomings or additional toolboxes, such that the wheel need not be reinvented. | 3 weeks | May 1 - May 24 |
| **2** | **Begin test assessments and complete Group 1 “required” tests.** | **4 weeks** | **May 25 - June 21** |
| 3 | Complete Group 2 “Strongly Recommended” tests. | 2 weeks | June 22 - July 5 |
| 4 | Complete Group 3 “Suggested” tests. | 3 weeks | July 6 - July 26 |
| 5 | Explore other tests and finalize documentation.| 3 weeks | July 27 - August 16 |

Therefore, the objectives for this deliverable are to assess each of the following tests in both the QARTOD data manual and in the toolbox itself.

1. Timing/Gap Test
2. Syntax Test
3. Location Test
4. Gross Range Test
    * Including similar Climatological Test (strongly recommended)
    * "Seasonal expectations" variation on the gross range test.
    * Allen et al. 2012 - Variability Generalixed Digital Environmental Model
    * Boyer et al. 2013 - WOD
5. Pressure Test
    * Including similar Density Inversion Test (suggested)
    * Pressure is looking for a monotonically ordered record.
    * This is distinguished by checking potential density $\sigma _{\theta}$ increases with increasing depth. Easily done with `gsw`.
    * Could bundle this with test 8 - Rate of Change.

In [4]:
#   Imports of required packages/modules/libraries for use in this deliverable's tweaks.
#   For handling more generic types (see https://docs.python.org/3/library/stdtypes.html)
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from collections.abc import Sequence    #   list, tuple, range are the three basic sequence types.
    from numbers import Real                #   Real numbers, including `int` and `float` but not complex or imaginary numbers.

import numpy as np
import xarray as xr

In [ ]:
#   Import a testing dataset
#   OG1 is OK?
fpath = "/home/aaron-mau/Data/OG1/delayed_SEA056_M102.nc"
ds = xr.load_dataset(fpath)

In [13]:
class QartodFlags:
    """Primary flags for QARTOD."""

    GOOD = 1
    UNKNOWN = 2
    SUSPECT = 3
    FAIL = 4
    MISSING = 9
FLAGS = QartodFlags  # Default name for all check modules
NOTEVAL_VALUE = QartodFlags.UNKNOWN

## Timing/Gap Test
* How is it described in the manual
* How is it represented in the code
* Does anything need to be done in either?

### Manual description of timing/gap test
It's a **required** test, so it's basically one of the most important checks that can be done on the data. Which makes sense - time is one of the dimensions of the data. If you don't know when the data was taken, it loses much of its value.

The manual describes it as "Check for arrival of data"
> Test determines that the most recent profile has been received within the expected time window
> (TIM_INC) and has the correct time stamp (TIM_STMP).
> 
> **Note:** For those gliders that do not update at regular intervals, a large value for TIM_STMP can be
> assigned. The gap check is not a solution for all timing errors. Data could be measured or received
> earlier than expected. This test does not address all clock drift/jump issues.

This is looking at expected time windows `(TIME_INC)` and has an appropriate time stamp `(TIM_STMP)`. In the example, `TIM_INC` is set to 6 hours.

Only 2 flags to assign: Either it passes the test (flag = 1) or it fails by not falling within the specified reporting range (flag = 4).

> `if NOW - TIM_STMP > TIM_INC, flag = 4`


### Code for timing/gap test
I did a quick search for the manual terms, and there's already a disconnect. There is no term for `TIM_STMP` or `TIM_INC` in the entire `ioos_qc` package. I think this is worth bringing up, as users will probably investigate in the same way that I am now. Looking for the terminology that is described in the manual within the code to see how the parameters match up.

Furthermore, there is no defined function or class within `qartod.py` that describes something representing a timing or gap test.
* `utils.py` has a "check_timestamps" function. However, the docstring explicitly says "This is not a QARTOD test, but rather a utility test to make sure **times are in the proper order** and optionally **do not have large gaps prior to processing the data**. It takes in a `max_time_interval` value and returns a boolean. It does not return flags - rather a single boolean of pass/fail.

**Question for Filipe**: We want to reuse code as much as possible, as defined in the GSoC project proposal. Does it make sense to "graduate" this function, or are we sacrificing utility? This `utils.check_timestamps` function is called in `test_utils.py`, so I don't know how it is actually used (if at all).
* It also *adds* a function of confirming that the times are in the proper order. This may not be desireable or in the scope of this description in the manual. **Question for Filipe**: Is that itself worthy of an additional test?

**Question for Filipe**: This is running on time, which is often a coordinate. Do you envision needing to do anything differently here? 

**Question for Filipe**: Is there a timestamp format that we should be expecting? Should it be homogenous in the notebook?

### Actions for timing/gap test
* Answer questions with Filipe, then act.
* I think that the manual's description of the test could be improved. "Check for arrival of data" $\rightarrow$ "Check data reporting period".

In [9]:
g = ds.TIME.to_numpy()

In [10]:
g

array(['2026-01-12T17:24:07.000000000', '2026-01-12T17:24:08.000000000',
       '2026-01-12T17:24:09.000000000', ...,
       '2026-02-10T11:17:06.000000000', '2026-02-10T11:17:07.000000000',
       '2026-02-10T11:17:08.000000000'],
      shape=(1690311,), dtype='datetime64[ns]')

In [14]:
len(g)

1690311

In [16]:
g.shape[0]

1690311

In [ ]:
from datetime import datetime
def time_gap_test(
    inp: Sequence[Real],
    fail_span: Real,
) -> np.ma.core.MaskedArray:
    """Determines that most recent data arrives within expected time window and has
    correct time stamp.
    
    Checks that time values are increasing within the expected period.


    Parameters
    ----------
    inp
        Input data (of timestamps) as a list-like sequence or array-like format.
    fail_span
        The FAIL threshold value, expressed in hours.
    
    Returns
    -------
    flag_arr
        A Numpy masked array of flag values equal in size to that of the input
    """
    now = datetime.now()
    flags = np.full((1, len(inp)-1))
    
    pass


2026-06-18 11:53:30.341231


In [12]:
QartodFlags

__main__.QartodFlags